# Daily Challenge: Custom Attention Mechanism & SMS Spam Classification
This notebook covers building a custom attention mechanism and comparing it to a pre-trained transformer (GPT-2) for SMS spam classification.

In [ ]:
# Part 1: Setup & Data Loading
!pip install --quiet datasets evaluate transformers[sentencepiece]

In [ ]:
import pandas as pd
from datasets import Dataset, load_dataset

In [ ]:
# Load the UCI SMS Spam dataset from Hugging Face hub
df = pd.read_parquet("hf://datasets/ucirvine/sms_spam/plain_text/train-00000-of-00001.parquet")
hf_dataset = Dataset.from_pandas(df)
train_ds = hf_dataset.select(range(4000))
val_ds = hf_dataset.select(range(4000, 5000))
df.head()

In [ ]:
# Part 2: Tokenization Setup
from transformers import GPT2Tokenizer
model_name = 'gpt2'
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def tokenize_fn(examples):
    return tokenizer(
        examples['sms'],
        padding='max_length',
        truncation=True,
        max_length=64
    )
train_tok = train_ds.map(tokenize_fn, batched=True)
val_tok = val_ds.map(tokenize_fn, batched=True)

In [ ]:
# Part 3: Pre-trained Model Setup
import torch
from transformers import GPT2ForSequenceClassification
model = GPT2ForSequenceClassification.from_pretrained(
    'gpt2',
    num_labels=2,
    pad_token_id=tokenizer.eos_token_id
)

In [ ]:
# Part 4: Custom Attention Implementation
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
class Attention(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.scale = embed_dim ** -0.5
    def forward(self, query, key, value, mask=None):
        scores = torch.matmul(query, key.transpose(-2, -1)) * self.scale
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        return torch.matmul(attn, value), attn

In [ ]:
class SimpleAttentionClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.attn = Attention(embed_dim)
        self.fc = nn.Linear(embed_dim, num_classes)
    def forward(self, x):
        embed = self.embedding(x)
        attn_output, _ = self.attn(embed, embed, embed)
        pooled = attn_output.mean(dim=1)
        return self.fc(pooled)

In [ ]:
def preprocess_for_attention(example):
    tokens = tokenizer.encode(
        example['sms'],
        max_length=64,
        truncation=True,
        padding='max_length'
    )
    return {'input_ids': tokens, 'label': example['label']}
train_ds_attn = train_ds.map(preprocess_for_attention)
val_ds_attn = val_ds.map(preprocess_for_attention)

In [ ]:
class SMSDataset(Dataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            'input_ids': torch.tensor(item['input_ids'], dtype=torch.long),
            'label': torch.tensor(item['label'], dtype=torch.long)
        }
train_loader = DataLoader(SMSDataset(train_ds_attn), batch_size=32, shuffle=True)
val_loader = DataLoader(SMSDataset(val_ds_attn), batch_size=32)

In [ ]:
vocab_size = tokenizer.vocab_size
embed_dim = 64
num_classes = 2
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
attn_model = SimpleAttentionClassifier(vocab_size, embed_dim, num_classes).to(device)
optimizer = torch.optim.Adam(attn_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
attn_model.train()
for batch in train_loader:
    inputs = batch['input_ids'].to(device)
    labels = batch['label'].to(device)
    optimizer.zero_grad()
    outputs = attn_model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
print('Custom Attention model trained on SMS dataset. Sample batch loss:', loss.item())

In [ ]:
import evaluate
import numpy as np
accuracy = evaluate.load('accuracy')
precision = evaluate.load('precision')
recall = evaluate.load('recall')
f1 = evaluate.load('f1')
def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy.compute(predictions=preds, references=labels)['accuracy'],
        'precision': precision.compute(predictions=preds, references=labels)['precision'],
        'recall': recall.compute(predictions=preds, references=labels)['recall'],
        'f1': f1.compute(predictions=preds, references=labels)['f1']
    }

In [ ]:
print('
📊 Evaluating GPT-2 Model...')
gpt2_preds = []
gpt2_labels = []
model.eval()
for ex in val_tok:
    inputs = torch.tensor(ex['input_ids']).unsqueeze(0).to(model.device)
    with torch.no_grad():
        logits = model(inputs).logits
    pred = torch.argmax(logits, dim=-1).cpu().item()
    gpt2_preds.append(pred)
    gpt2_labels.append(ex['label'])
gpt2_metrics = {
    'accuracy': accuracy.compute(predictions=gpt2_preds, references=gpt2_labels)['accuracy'],
    'precision': precision.compute(predictions=gpt2_preds, references=gpt2_labels)['precision'],
    'recall': recall.compute(predictions=gpt2_preds, references=gpt2_labels)['recall'],
    'f1': f1.compute(predictions=gpt2_preds, references=gpt2_labels)['f1']
}
print('GPT-2 Metrics:', gpt2_metrics)
print('
📊 Evaluating Custom Attention Model...')
attn_preds = []
attn_labels = []
attn_model.eval()
for batch in val_loader:
    inputs = batch['input_ids'].to(device)
    labels = batch['label'].to(device)
    with torch.no_grad():
        outputs = attn_model(inputs)
        preds = torch.argmax(outputs, dim=1)
    attn_preds.extend(preds.cpu().tolist())
    attn_labels.extend(labels.cpu().tolist())
attn_metrics = {
    'accuracy': accuracy.compute(predictions=attn_preds, references=attn_labels)['accuracy'],
    'precision': precision.compute(predictions=attn_preds, references=attn_labels)['precision'],
    'recall': recall.compute(predictions=attn_preds, references=attn_labels)['recall'],
    'f1': f1.compute(predictions=attn_preds, references=attn_labels)['f1']
}
print('Attention Model Metrics:', attn_metrics)

# Reflection Questions
1. **Roles of Query, Key, and Value in Attention:**
- The query represents the current position or token seeking information.
- Keys contain information about all positions/tokens in the sequence.
- Values are the actual data to be aggregated, weighted by attention scores.

2. **Why use a scaling factor in dot-product attention?**
- As dimensionality increases, dot products can become very large, causing softmax to have extremely small gradients.
- The scaling factor (1/sqrt(d_k)) normalizes the scores, stabilizing training and improving gradient flow.

3. **Self-attention vs. RNNs:**
- Self-attention processes all tokens in parallel, while RNNs process sequentially.
- Self-attention can model long-range dependencies more efficiently.
- Self-attention is more computationally efficient and easier to train than RNNs.

4. **Performance Analysis:**
- GPT-2 likely performs better due to pretraining on large datasets and a more complex architecture.
- The custom attention model is simpler and may underfit, but is easier to interpret and faster to train.
- Improvements for the custom model could include deeper layers, regularization, or more advanced attention mechanisms.